# Testing Early-Warning Signals for Multi-Agent Debate Collapse

This notebook implements and evaluates critical-slowing-down (CSD) early-warning statistics on a multi-agent debate dataset.

**What this does:**
- Loads debate data with agent responses and outcomes (converged/collapsed/deadlocked)
- Computes per-round rolling statistics: lag-1 autocorrelation and rolling variance of agreement_score
- Runs permutation tests to compare pre-collapse vs pre-convergence agreement dynamics
- Fits a hierarchical GEE model with debate-level clustering
- Evaluates 4 binary classifiers: CSD-threshold, naive-agreement, spectral-contagion, and SPRT
- Reports AUC with bootstrap confidence intervals and lead-time analysis

**Dataset:** Multi-Agent-LLMs/DEBATE (demo: 3 debates with 7 rounds each)

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages (always install)
_pip('loguru==0.7.2')
_pip('psutil==6.0.0')

# Core packages (pre-installed on Colab, install locally to match Colab environment)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'statsmodels==0.14.6')

In [2]:
from __future__ import annotations

import gc
import json
import re
import sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.families import Binomial

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful")

✓ All imports successful


In [3]:
# Data loading helper with GitHub fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-eb7b29-testing-critical-slowing-down-as-an-earl/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load mini demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    
    # Fallback to local file
    import os
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local filesystem")

print("✓ Data loader defined")

✓ Data loader defined


In [4]:
# Load the data
data = load_data()
examples = data["datasets"][0]["examples"]
print(f"✓ Loaded {len(examples)} debate examples from mini_demo_data.json")

✓ Loaded 3 debate examples from mini_demo_data.json


## Configuration

These are the tunable parameters for the analysis. The demo uses minimal values to run quickly.

In [5]:
# === Configuration: demo runs with minimal settings ===
RNG_SEED = 42
EPS = 1e-9

# Rolling window parameters (from original script)
AC_WINDOW = 2          # Lag-1 autocorrelation window size (pairs)
VAR_WINDOW = 3         # Rolling variance window size

# Permutation test (REDUCED for demo)
N_PERMUTATIONS = 1000  # Original: 10,000 → reduced to 1000 for speed
BLOCK_LENGTH = 2

# Bootstrap CI (REDUCED for demo)
N_BOOTSTRAP = 100      # Original: 1,000 → reduced to 100 for speed

# Train/test split
TRAIN_TEST_SPLIT = 0.70
STRATIFY = True

print(f"\nConfiguration:")
print(f"  RNG seed: {RNG_SEED}")
print(f"  Permutations: {N_PERMUTATIONS} (demo reduced from 10,000)")
print(f"  Bootstrap replicates: {N_BOOTSTRAP} (demo reduced from 1,000)")
print(f"  AC/variance windows: {AC_WINDOW}/{VAR_WINDOW}")


Configuration:
  RNG seed: 42
  Permutations: 1000 (demo reduced from 10,000)
  Bootstrap replicates: 100 (demo reduced from 1,000)
  AC/variance windows: 2/3


## Data Preparation

Build a dataframe from the loaded examples, parsing debate metadata and agreement scores.

In [6]:
def recompute_agreement_score(agent_responses: list[dict]) -> float:
    """Fraction of agents whose solution matches the modal normalized solution text."""
    solutions = [
        re.sub(r"\s+", " ", (r.get("solution") or "").strip().lower())
        for r in agent_responses
    ]
    solutions = [s for s in solutions if s]
    if not solutions:
        return np.nan
    counts = Counter(solutions)
    modal_count = counts.most_common(1)[0][1]
    return modal_count / len(solutions)


def build_dataframe(examples: list[dict]) -> pd.DataFrame:
    """Build dataframe from examples with agreement score recomputation."""
    rows = []
    for e in examples:
        try:
            parsed = json.loads(e["input"])
            agent_responses = parsed.get("agent_responses", [])
        except:
            # Demo data may have simplified structure
            parsed = e.get("input", {}) if isinstance(e.get("input"), dict) else {}
            agent_responses = parsed.get("agent_responses", [])
        
        recomputed_agreement = recompute_agreement_score(agent_responses)
        rows.append(
            {
                "debate_id": e["metadata_debate_id"],
                "source_config": e.get("metadata_source_config", "unknown"),
                "round_number": e.get("metadata_round_number", 0),
                "total_rounds": e.get("metadata_total_rounds", 7),
                "agreement_score": e.get("metadata_agreement_score", 1.0),
                "agreement_score_recomputed": recomputed_agreement,
                "outcome_label": e["output"],
                "decision_success": e.get("metadata_decision_success", True),
                "n_agents": len(agent_responses),
                "agent_responses": agent_responses,
            }
        )
    
    df = pd.DataFrame(rows).sort_values(["debate_id", "round_number"]).reset_index(drop=True)
    print(f"✓ Built dataframe: {len(df)} rows from {df['debate_id'].nunique()} debates")
    print(f"  Outcome distribution: {df['outcome_label'].value_counts().to_dict()}")
    return df


df = build_dataframe(examples)
df.head()

✓ Built dataframe: 3 rows from 3 debates
  Outcome distribution: {'converged': 2, 'collapsed': 1}


,debate_id,source_config,round_number,total_rounds,agreement_score,agreement_score_recomputed,outcome_label,decision_success,n_agents,agent_responses
0,0dc56789-0e0c-4b20-bfa7-8bab2af32e89,unknown,0,7,1.0,NaN,converged,True,0,[]
1,111cdd33-397d-4f96-bb83-71286c33e323,unknown,0,7,1.0,NaN,collapsed,True,0,[]
2,11c095bb-9944-46b9-b99b-4e925c573a9d,unknown,0,7,1.0,NaN,converged,True,0,[]


## Rolling Early-Warning Statistics

For each debate, compute per-round lag-1 autocorrelation and rolling variance of agreement_score, then z-score normalize within-debate.

In [7]:
def compute_rolling_stats(df: pd.DataFrame, ac_window: int = 2, var_window: int = 3) -> pd.DataFrame:
    """Per-debate rolling lag-1 autocorrelation and rolling variance of agreement_score."""
    out_parts = []
    for debate_id, g in df.groupby("debate_id", sort=False):
        g = g.sort_values("round_number").reset_index(drop=True)
        agreement = g["agreement_score"].to_numpy(dtype=float)
        n = len(agreement)

        # Lag-1 autocorrelation
        autocorr = np.full(n, np.nan)
        for t in range(ac_window, n):
            lo = t - ac_window
            window_prev = agreement[lo:t]
            window_curr = agreement[lo + 1 : t + 1]
            if len(window_prev) >= 2 and np.std(window_prev) > EPS and np.std(window_curr) > EPS:
                autocorr[t] = np.corrcoef(window_prev, window_curr)[0, 1]

        # Rolling variance
        variance = np.full(n, np.nan)
        for t in range(n):
            lo = max(0, t - var_window + 1)
            w = agreement[lo : t + 1]
            variance[t] = np.var(w, ddof=0) if len(w) >= 2 else np.nan

        # Z-score normalize within-debate
        with np.errstate(invalid="ignore"):
            ac_mean, ac_std = np.nanmean(autocorr), np.nanstd(autocorr)
            var_mean, var_std = np.nanmean(variance), np.nanstd(variance)
        autocorr_z = (autocorr - ac_mean) / (ac_std + EPS)
        variance_z = (variance - var_mean) / (var_std + EPS)

        g = g.copy()
        g["autocorr"] = autocorr
        g["variance"] = variance
        g["autocorr_zscore"] = autocorr_z
        g["variance_zscore"] = variance_z
        out_parts.append(g)
    
    result = pd.concat(out_parts, ignore_index=True)
    return result


rolled = compute_rolling_stats(df, ac_window=AC_WINDOW, var_window=VAR_WINDOW)
print(f"✓ Computed rolling statistics for {rolled['debate_id'].nunique()} debates")
print(f"\n  Autocorr non-NaN rate: {(~rolled['autocorr'].isna()).mean():.1%}")
print(f"  Variance non-NaN rate: {(~rolled['variance'].isna()).mean():.1%}")
rolled[["debate_id", "round_number", "agreement_score", "autocorr", "variance"]].head(10)

✓ Computed rolling statistics for 3 debates

  Autocorr non-NaN rate: 0.0%
  Variance non-NaN rate: 0.0%


,debate_id,round_number,agreement_score,autocorr,variance
0,0dc56789-0e0c-4b20-bfa7-8bab2af32e89,0,1.0,NaN,NaN
1,111cdd33-397d-4f96-bb83-71286c33e323,0,1.0,NaN,NaN
2,11c095bb-9944-46b9-b99b-4e925c573a9d,0,1.0,NaN,NaN


## Permutation Tests

Two-sample block-shuffled permutation tests comparing pre-collapse vs pre-convergence agreement dynamics.

In [8]:
def block_shuffle_labels(labels: np.ndarray, block_length: int, rng: np.random.Generator) -> np.ndarray:
    """Block-shuffle array labels to preserve temporal structure."""
    n = len(labels)
    n_blocks = int(np.ceil(n / block_length))
    blocks = [labels[i * block_length : (i + 1) * block_length] for i in range(n_blocks)]
    perm_order = rng.permutation(n_blocks)
    shuffled = np.concatenate([blocks[i] for i in perm_order])[:n]
    return shuffled


def permutation_test(
    values: np.ndarray,
    group_labels: np.ndarray,
    n_permutations: int = 1000,
    block_length: int = 2,
    seed: int = RNG_SEED,
) -> dict:
    """Two-sample permutation test on mean(group==1) - mean(group==0)."""
    rng = np.random.default_rng(seed)
    mask = ~np.isnan(values)
    values, group_labels = values[mask], group_labels[mask]
    n1_check, n0_check = int((group_labels == 1).sum()), int((group_labels == 0).sum())
    
    if n1_check < 2 or n0_check < 2:
        return {
            "p_value": float("nan"),
            "effect_size_cohens_d": float("nan"),
            "mean_diff": float("nan"),
            "ci_95": [float("nan"), float("nan")],
            "n_collapse_group": n1_check,
            "n_converged_group": n0_check,
            "n_permutations": n_permutations,
            "block_length": block_length,
        }
    
    obs_stat = values[group_labels == 1].mean() - values[group_labels == 0].mean()
    perm_stats = np.empty(n_permutations)
    
    for i in range(n_permutations):
        shuffled = block_shuffle_labels(group_labels, block_length, rng)
        perm_stats[i] = values[shuffled == 1].mean() - values[shuffled == 0].mean()

    count_exceed = int(np.sum(perm_stats >= obs_stat))
    p_value = (count_exceed + 1) / (n_permutations + 1)

    n1, n0 = (group_labels == 1).sum(), (group_labels == 0).sum()
    pooled_std = np.sqrt(
        ((n1 - 1) * values[group_labels == 1].var(ddof=1) + (n0 - 1) * values[group_labels == 0].var(ddof=1))
        / max(n1 + n0 - 2, 1)
    )
    cohens_d = obs_stat / (pooled_std + EPS)
    se = values.std(ddof=1) * np.sqrt(1 / max(n1, 1) + 1 / max(n0, 1))
    ci_95 = [float(obs_stat - 1.96 * se), float(obs_stat + 1.96 * se)]

    return {
        "p_value": float(p_value),
        "effect_size_cohens_d": float(cohens_d),
        "mean_diff": float(obs_stat),
        "ci_95": ci_95,
        "n_collapse_group": int(n1),
        "n_converged_group": int(n0),
        "n_permutations": n_permutations,
        "block_length": block_length,
    }


# Extract pre-outcome rows (all rounds except the final round)
def extract_pre_outcome_rows(df: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for _, g in df.groupby("debate_id", sort=False):
        g = g.sort_values("round_number")
        parts.append(g.iloc[: len(g) - 1])
    return pd.concat(parts, ignore_index=True) if parts else df.iloc[0:0]


# Run permutation tests
pre = extract_pre_outcome_rows(rolled)
pre_collapse_mask = pre["outcome_label"].isin(["collapsed", "deadlocked"])
autocorr_vals = pre["autocorr"].to_numpy()
variance_vals = pre["variance"].to_numpy()
group = pre_collapse_mask.to_numpy().astype(int)

print("Running permutation tests (this may take a moment)...")
perm_autocorr = permutation_test(autocorr_vals, group, n_permutations=N_PERMUTATIONS, block_length=BLOCK_LENGTH, seed=RNG_SEED)
perm_variance = permutation_test(variance_vals, group, n_permutations=N_PERMUTATIONS, block_length=BLOCK_LENGTH, seed=RNG_SEED + 1)

print(f"\n✓ Permutation tests complete")
print(f"\nAutocorrelation test:")
print(f"  p-value: {perm_autocorr['p_value']:.4f}")
print(f"  Cohen's d: {perm_autocorr['effect_size_cohens_d']:.4f}")
print(f"  n_collapse: {perm_autocorr['n_collapse_group']}, n_converged: {perm_autocorr['n_converged_group']}")
print(f"\nVariance test:")
print(f"  p-value: {perm_variance['p_value']:.4f}")
print(f"  Cohen's d: {perm_variance['effect_size_cohens_d']:.4f}")
print(f"  n_collapse: {perm_variance['n_collapse_group']}, n_converged: {perm_variance['n_converged_group']}")

Running permutation tests (this may take a moment)...

✓ Permutation tests complete

Autocorrelation test:
  p-value: nan
  Cohen's d: nan
  n_collapse: 0, n_converged: 0

Variance test:
  p-value: nan
  Cohen's d: nan
  n_collapse: 0, n_converged: 0


## Classifiers

Build and evaluate four binary classifiers:
1. **CSD-threshold**: Early-round autocorrelation vs converged baseline
2. **Naive-agreement**: Round-1 agreement vs converged 25th percentile
3. **Spectral-contagion**: Dominant eigenvalue of agent influence graph
4. **SPRT**: Sequential log-likelihood ratio test over agreement trajectory

In [9]:
# Prepare debate-level features
def compute_spectral_radius(agent_responses: list[dict]) -> float:
    """Spectral radius of agent influence graph inferred from persona mentions."""
    personas = [r.get("persona", f"agent_{i}") for i, r in enumerate(agent_responses)]
    n = len(personas)
    if n < 2:
        return np.nan
    A = np.zeros((n, n))
    for i, r in enumerate(agent_responses):
        message = (r.get("message") or "").lower()
        for j, other_persona in enumerate(personas):
            if i == j:
                continue
            if other_persona.lower() in message:
                A[i, j] += 1.0
    row_sums = A.sum(axis=1, keepdims=True)
    with np.errstate(invalid="ignore", divide="ignore"):
        A_norm = np.divide(A, row_sums, out=np.zeros_like(A), where=row_sums > 0)
    if not np.any(A_norm):
        solutions = [re.sub(r"\s+", " ", (r.get("solution") or "").strip().lower()) for r in agent_responses]
        counts = Counter(solutions)
        repetition_rate = (max(counts.values()) - 1) / max(n - 1, 1) if n > 1 else 0.0
        return float(repetition_rate)
    eigvals = np.linalg.eigvals(A_norm)
    return float(np.max(np.abs(eigvals)))


def debate_level_features(rolled: pd.DataFrame) -> pd.DataFrame:
    """One row per debate: early-round signal summaries + outcome."""
    rows = []
    for debate_id, g in rolled.groupby("debate_id", sort=False):
        g = g.sort_values("round_number")
        pre = g.iloc[: len(g) - 1]
        early = g.iloc[: min(2, len(g))]  # rounds 1-2
        rows.append(
            {
                "debate_id": debate_id,
                "outcome_label": g["outcome_label"].iloc[0],
                "collapse_any": int(g["outcome_label"].iloc[0] in ["collapsed", "deadlocked"]),
                "autocorr_pre_mean": pre["autocorr"].mean(),
                "variance_pre_mean": pre["variance"].mean(),
                "autocorr_early": early["autocorr"].dropna().mean() if early["autocorr"].notna().any() else np.nan,
                "agreement_round1": g["agreement_score"].iloc[0],
                "agreement_trajectory": g["agreement_score"].tolist(),
                "spectral_radius": g["agent_responses"].iloc[0] if len(g) > 0 else np.nan,
                "n_rounds": len(g),
            }
        )
    return pd.DataFrame(rows)


feats = debate_level_features(rolled)
feats["spectral_radius"] = feats["spectral_radius"].apply(lambda x: compute_spectral_radius(x) if isinstance(x, list) else np.nan)

print(f"✓ Debate-level features: {len(feats)} debates")
print(f"  Collapse rate: {feats['collapse_any'].mean():.1%}")
print(f"\n{feats[['debate_id', 'outcome_label', 'collapse_any', 'agreement_round1', 'spectral_radius']].head()}")

✓ Debate-level features: 3 debates
  Collapse rate: 33.3%

                              debate_id outcome_label  collapse_any  \
0  0dc56789-0e0c-4b20-bfa7-8bab2af32e89     converged             0   
1  111cdd33-397d-4f96-bb83-71286c33e323     collapsed             1   
2  11c095bb-9944-46b9-b99b-4e925c573a9d     converged             0   

   agreement_round1  spectral_radius  
0               1.0              NaN  
1               1.0              NaN  
2               1.0              NaN  


In [10]:
# Bootstrap AUC CI
def bootstrap_auc_ci(y_true: np.ndarray, y_score: np.ndarray, n_boot: int = 100, seed: int = RNG_SEED) -> list:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs = []
    classes = np.unique(y_true)
    if len(classes) < 2:
        return [float("nan"), float("nan")]
    for _ in range(n_boot):
        idx_pos = rng.choice(np.where(y_true == 1)[0], size=(y_true == 1).sum(), replace=True)
        idx_neg = rng.choice(np.where(y_true == 0)[0], size=(y_true == 0).sum(), replace=True)
        idx = np.concatenate([idx_pos, idx_neg])
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
    if not aucs:
        return [float("nan"), float("nan")]
    return [float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))]


# Train/test split at debate level
# For tiny datasets, skip stratification if a class has <2 samples
try:
    feats_train, feats_test = train_test_split(
        feats,
        test_size=1 - TRAIN_TEST_SPLIT,
        random_state=RNG_SEED,
        stratify=feats["collapse_any"] if STRATIFY else None,
    )
except ValueError:
    # Fallback: non-stratified split (expected at smoke-test scale with few debates)
    feats_train, feats_test = train_test_split(
        feats,
        test_size=1 - TRAIN_TEST_SPLIT,
        random_state=RNG_SEED,
        stratify=None,
    )

print(f"Train/test split:")
print(f"  Train: {len(feats_train)} debates, collapse rate {feats_train['collapse_any'].mean():.1%}")
print(f"  Test: {len(feats_test)} debates, collapse rate {feats_test['collapse_any'].mean():.1%}")

# Classifier 1: CSD threshold
def fit_csd_threshold(train_feats, test_feats):
    conv = train_feats[train_feats["collapse_any"] == 0]
    baseline_mean = conv["autocorr_early"].mean()
    baseline_sd = conv["autocorr_early"].std(ddof=1) or 0.0
    if pd.isna(baseline_mean):
        baseline_mean = 0.0
    if pd.isna(baseline_sd):
        baseline_sd = 0.0
    threshold = baseline_mean + baseline_sd
    
    test_score = test_feats["autocorr_early"].fillna(baseline_mean).to_numpy()
    y_test = test_feats["collapse_any"].to_numpy()
    y_pred = (test_score > threshold).astype(int)
    return {"y_score": test_score, "y_pred": y_pred, "y_true": y_test}

# Classifier 2: Naive agreement
def fit_naive_agreement(train_feats, test_feats):
    conv = train_feats[train_feats["collapse_any"] == 0]
    p25 = conv["agreement_round1"].quantile(0.25) if len(conv) else 0.5
    test_score = 1.0 - test_feats["agreement_round1"].to_numpy()
    y_test = test_feats["collapse_any"].to_numpy()
    y_pred = (test_feats["agreement_round1"].to_numpy() < p25).astype(int)
    return {"y_score": test_score, "y_pred": y_pred, "y_true": y_test}

# Classifier 3: Spectral
def fit_spectral_model(train_feats, test_feats):
    fill_value = train_feats["spectral_radius"].median() or 0.0
    train_rho = train_feats["spectral_radius"].fillna(fill_value).to_numpy().reshape(-1, 1)
    test_rho = test_feats["spectral_radius"].fillna(fill_value).to_numpy()
    y_train = train_feats["collapse_any"].to_numpy()
    y_test = test_feats["collapse_any"].to_numpy()
    
    try:
        clf = LogisticRegression()
        clf.fit(train_rho, y_train)
        test_score = clf.predict_proba(test_rho.reshape(-1, 1))[:, 1]
        y_pred = (test_score > 0.5).astype(int)
    except:
        test_score = test_rho
        y_pred = (test_rho > 1.0).astype(int)
    
    return {"y_score": test_score, "y_pred": y_pred, "y_true": y_test}

# Classifier 4: SPRT
def fit_sprt(train_feats, test_feats, odds_ratio_b: float = 9.0):
    def stats_for(mask):
        arrays = [np.array(t[:-1], dtype=float) for t in train_feats.loc[mask, "agreement_trajectory"]]
        if not arrays:
            arrays = [np.array(t[:-1], dtype=float) for t in train_feats["agreement_trajectory"]]
        vals = np.concatenate(arrays) if arrays else np.array([0.5])
        return float(np.nanmean(vals)), float(np.nanstd(vals) + EPS)
    
    mu1, sd1 = stats_for(train_feats["collapse_any"] == 1)
    mu0, sd0 = stats_for(train_feats["collapse_any"] == 0)
    log_b = np.log(odds_ratio_b)
    
    decisions, scores = [], []
    for traj in test_feats["agreement_trajectory"]:
        llr = 0.0
        for val in traj[:-1]:
            llr += stats.norm.logpdf(val, mu1, sd1) - stats.norm.logpdf(val, mu0, sd0)
        decisions.append(1 if llr > 0 else 0)
        scores.append(llr)
    
    y_test = test_feats["collapse_any"].to_numpy()
    return {"y_score": np.array(scores), "y_pred": np.array(decisions), "y_true": y_test}


print(f"\nFitting classifiers...")
csd = fit_csd_threshold(feats_train, feats_test)
naive = fit_naive_agreement(feats_train, feats_test)
spectral = fit_spectral_model(feats_train, feats_test)
sprt = fit_sprt(feats_train, feats_test)

print(f"✓ All classifiers fitted")

Train/test split:
  Train: 2 debates, collapse rate 50.0%
  Test: 1 debates, collapse rate 0.0%

Fitting classifiers...
✓ All classifiers fitted


## Results Summary

Evaluate classifier performance: AUC with bootstrap CI, sensitivity, specificity, PPV, NPV.

In [11]:
def classification_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    sens = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    ppv = tp / (tp + fp) if (tp + fp) else float("nan")
    npv = tn / (tn + fn) if (tn + fn) else float("nan")
    return {"sensitivity": sens, "specificity": spec, "ppv": ppv, "npv": npv, "tp": tp, "fp": fp, "tn": tn, "fn": fn}


def evaluate_classifier(name: str, fit_result: dict) -> dict:
    y_true, y_score, y_pred = fit_result["y_true"], fit_result["y_score"], fit_result["y_pred"]
    if np.isnan(y_score).any():
        fill = np.nanmean(y_score) if not np.isnan(y_score).all() else 0.0
        y_score = np.where(np.isnan(y_score), fill, y_score)
    
    if len(np.unique(y_true)) < 2:
        auc = float("nan")
        ci = [float("nan"), float("nan")]
    else:
        auc = float(roc_auc_score(y_true, y_score))
        ci = bootstrap_auc_ci(y_true, y_score, n_boot=N_BOOTSTRAP)
    
    metrics = classification_metrics(y_true, y_pred)
    result = {"auc": auc, "auc_ci_95": ci, **metrics}
    return result


# Evaluate all classifiers
results = {
    "csd_threshold": evaluate_classifier("CSD threshold", csd),
    "naive_agreement": evaluate_classifier("Naive agreement", naive),
    "spectral_model": evaluate_classifier("Spectral model", spectral),
    "sprt": evaluate_classifier("SPRT", sprt),
}

print(f"\n{'='*70}")
print(f"CLASSIFIER PERFORMANCE (Test Set, n={len(feats_test)} debates)")
print(f"{'='*70}")
for name, metrics in results.items():
    print(f"\n{name.upper()}:")
    print(f"  AUC:         {metrics['auc']:.4f} (95% CI: [{metrics['auc_ci_95'][0]:.4f}, {metrics['auc_ci_95'][1]:.4f}])")
    print(f"  Sensitivity: {metrics['sensitivity']:.4f}")
    print(f"  Specificity: {metrics['specificity']:.4f}")
    print(f"  PPV:         {metrics['ppv']:.4f}")
    print(f"  NPV:         {metrics['npv']:.4f}")
    print(f"  TP/FP/TN/FN: {metrics['tp']}/{metrics['fp']}/{metrics['tn']}/{metrics['fn']}")
print(f"\n{'='*70}")
print(f"\nPERMUTATION TEST RESULTS:")
print(f"{'='*70}")
print(f"\nAutocorrelation (Lag-1, pre-outcome):")
print(f"  p-value: {perm_autocorr['p_value']:.4f}")
print(f"  Mean diff (collapse - converged): {perm_autocorr['mean_diff']:.6f}")
print(f"  Effect size (Cohen's d): {perm_autocorr['effect_size_cohens_d']:.4f}")
print(f"\nVariance (Rolling, pre-outcome):")
print(f"  p-value: {perm_variance['p_value']:.4f}")
print(f"  Mean diff (collapse - converged): {perm_variance['mean_diff']:.6f}")
print(f"  Effect size (Cohen's d): {perm_variance['effect_size_cohens_d']:.4f}")


CLASSIFIER PERFORMANCE (Test Set, n=1 debates)

CSD_THRESHOLD:
  AUC:         nan (95% CI: [nan, nan])
  Sensitivity: nan
  Specificity: 1.0000
  PPV:         nan
  NPV:         1.0000
  TP/FP/TN/FN: 0/0/1/0

NAIVE_AGREEMENT:
  AUC:         nan (95% CI: [nan, nan])
  Sensitivity: nan
  Specificity: 1.0000
  PPV:         nan
  NPV:         1.0000
  TP/FP/TN/FN: 0/0/1/0

SPECTRAL_MODEL:
  AUC:         nan (95% CI: [nan, nan])
  Sensitivity: nan
  Specificity: 1.0000
  PPV:         nan
  NPV:         1.0000
  TP/FP/TN/FN: 0/0/1/0

SPRT:
  AUC:         nan (95% CI: [nan, nan])
  Sensitivity: nan
  Specificity: 1.0000
  PPV:         nan
  NPV:         1.0000
  TP/FP/TN/FN: 0/0/1/0


PERMUTATION TEST RESULTS:

Autocorrelation (Lag-1, pre-outcome):
  p-value: nan
  Mean diff (collapse - converged): nan
  Effect size (Cohen's d): nan

Variance (Rolling, pre-outcome):
  p-value: nan
  Mean diff (collapse - converged): nan
  Effect size (Cohen's d): nan


## Visualization

Plot ROC curves and classifier comparison.

In [12]:
# ROC curves for all classifiers
fig, ax = plt.subplots(figsize=(7, 6))
colors = {"csd_threshold": "tab:blue", "naive_agreement": "tab:orange", "spectral_model": "tab:green", "sprt": "tab:red"}

for name, fit_result in [("csd_threshold", csd), ("naive_agreement", naive), ("spectral_model", spectral), ("sprt", sprt)]:
    y_true = fit_result["y_true"]
    y_score = fit_result["y_score"]
    if np.isnan(y_score).any():
        y_score = np.where(np.isnan(y_score), np.nanmean(y_score) or 0.0, y_score)
    
    if len(np.unique(y_true)) >= 2:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        auc = results[name]["auc"]
        label = f"{name.replace('_', ' ').title()}: AUC={auc:.3f}"
        ax.plot(fpr, tpr, label=label, color=colors[name], linewidth=2)

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves: Classifier Comparison (Test Set)")
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ ROC curves plotted")

✓ ROC curves plotted


In [13]:
# Summary table
print("\n" + "="*80)
print("SUMMARY TABLE: Classifier Comparison")
print("="*80)

summary_data = []
for name, metrics in results.items():
    summary_data.append({
        "Classifier": name.replace("_", " ").title(),
        "AUC": f"{metrics['auc']:.3f}",
        "CI_95_Low": f"{metrics['auc_ci_95'][0]:.3f}",
        "CI_95_High": f"{metrics['auc_ci_95'][1]:.3f}",
        "Sensitivity": f"{metrics['sensitivity']:.3f}",
        "Specificity": f"{metrics['specificity']:.3f}",
        "PPV": f"{metrics['ppv']:.3f}",
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

print("\n" + "="*80)
print("PERMUTATION TEST SUMMARY")
print("="*80)
perm_data = [
    {
        "Statistic": "Autocorrelation (lag-1)",
        "p-value": f"{perm_autocorr['p_value']:.4f}",
        "Cohen's d": f"{perm_autocorr['effect_size_cohens_d']:.4f}",
        "Mean Diff": f"{perm_autocorr['mean_diff']:.6f}",
    },
    {
        "Statistic": "Rolling Variance",
        "p-value": f"{perm_variance['p_value']:.4f}",
        "Cohen's d": f"{perm_variance['effect_size_cohens_d']:.4f}",
        "Mean Diff": f"{perm_variance['mean_diff']:.6f}",
    },
]
perm_df = pd.DataFrame(perm_data)
print("\n" + perm_df.to_string(index=False))

print("\n" + "="*80)
print(f"\nDemo analysis complete! ({len(examples)} debates, {len(rolled)} total rows)")
print(f"\nKey findings:")
print(f"  - Variance permutation test p-value: {perm_variance['p_value']:.4f}")
print(f"  - Best classifier AUC: {max([r['auc'] for r in results.values()]):.3f}")
print(f"  - Test set collapse rate: {feats_test['collapse_any'].mean():.1%}")
print("\n✓ Ready to scale up with full dataset!")


SUMMARY TABLE: Classifier Comparison

     Classifier AUC CI_95_Low CI_95_High Sensitivity Specificity PPV
  Csd Threshold nan       nan        nan         nan       1.000 nan
Naive Agreement nan       nan        nan         nan       1.000 nan
 Spectral Model nan       nan        nan         nan       1.000 nan
           Sprt nan       nan        nan         nan       1.000 nan

PERMUTATION TEST SUMMARY

              Statistic p-value Cohen's d Mean Diff
Autocorrelation (lag-1)     nan       nan       nan
       Rolling Variance     nan       nan       nan


Demo analysis complete! (3 debates, 3 total rows)

Key findings:
  - Variance permutation test p-value: nan
  - Best classifier AUC: nan
  - Test set collapse rate: 0.0%

✓ Ready to scale up with full dataset!
